# Scanning & Filtering Playground

Explore the TidyForge scanning and filtering primitives interactively.

This is read-only — nothing modifies your files. Use this to:
- Scan directories and see what's there
- Test filter combinations before using them with `organize`
- Inspect file metadata
- Group files by date, extension, or folder

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
src_dirs = [
    "packages/core/src",
    "packages/fs-indexer/src",
    "packages/metadata/src",
    "packages/media-grouping/src",
    "packages/duplicate-detection/src",
    "packages/rename-engine/src",
    "packages/job-runner/src",
    "packages/ui-common/src",
    "apps/media-curator/src",
    "apps/disk-atlas/src",
    "apps/rename-studio/src",
    "apps/control-center/src",
]
for d in src_dirs:
    p = str(ROOT / d)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root: {ROOT}")

## 1. Basic Scan

Scan a directory and see all files.

In [ ]:
from tidyforge.fs_indexer import scan_directory
from tidyforge.ui_common import format_size

# ---- EDIT THIS ----
TARGET_DIR = Path.home() / "Pictures"
MAX_DEPTH = 1   # None for unlimited, 0 for immediate children only
# -------------------

all_entries = list(scan_directory(TARGET_DIR, max_depth=MAX_DEPTH))
print(f"Found {len(all_entries)} files in {TARGET_DIR}")

# Quick summary
total_size = sum(e.size for e in all_entries)
print(f"Total size: {format_size(total_size)}")

In [ ]:
# Show first 20 files
for e in all_entries[:20]:
    print(f"  {e.suffix:6s}  {e.size_human:>8s}  {e.modified:%Y-%m-%d %H:%M}  {e.name}")
if len(all_entries) > 20:
    print(f"  ... and {len(all_entries) - 20} more")

## 2. Filter by Extension

In [ ]:
from tidyforge.fs_indexer import ExtensionFilter

# ---- EDIT THIS ----
INCLUDE_EXTS = {".jpg", ".jpeg", ".png"}
# -------------------

ext_filter = ExtensionFilter(include=INCLUDE_EXTS)
filtered = list(scan_directory(TARGET_DIR, filters=[ext_filter], max_depth=MAX_DEPTH))
print(f"{len(filtered)} files match extensions {INCLUDE_EXTS}")
for e in filtered[:10]:
    print(f"  {e.name}")

## 3. Filter by Name Pattern (Prefix, Suffix, Regex)

In [ ]:
import re

from tidyforge.fs_indexer import PatternFilter

# ---- Try each one ----

# Prefix: files whose name starts with "IMG_"
prefix = "IMG_"
pf = PatternFilter(pattern=f"^{re.escape(prefix)}")
matches = [e for e in all_entries if pf(e)]
print(f"Prefix '{prefix}': {len(matches)} matches")
for e in matches[:5]:
    print(f"  {e.name}")

In [ ]:
# Suffix: files whose stem ends with "_edited" (before the extension)
suffix = "_edited"
sf = PatternFilter(pattern=f"{re.escape(suffix)}\\.[^.]*$")
matches = [e for e in all_entries if sf(e)]
print(f"Suffix '{suffix}': {len(matches)} matches")
for e in matches[:5]:
    print(f"  {e.name}")

In [ ]:
# Regex: any custom pattern against the full filename
pattern = r"\d{8}"  # e.g. files containing 8 consecutive digits (like a date)
rf = PatternFilter(pattern=pattern)
matches = [e for e in all_entries if rf(e)]
print(f"Regex '{pattern}': {len(matches)} matches")
for e in matches[:5]:
    print(f"  {e.name}")

## 4. Combine Filters

In [ ]:
from tidyforge.fs_indexer import CompositeFilter

# AND: all filters must match
combined_and = CompositeFilter(
    filters=[
        ExtensionFilter(include={".jpg", ".jpeg"}),
        PatternFilter(pattern=r"^IMG_"),
    ],
    mode="and",
)
matches = [e for e in all_entries if combined_and(e)]
print(f"AND filter (jpg + IMG_ prefix): {len(matches)} matches")

# OR: any filter can match
combined_or = CompositeFilter(
    filters=[
        PatternFilter(pattern=r"^IMG_"),
        PatternFilter(pattern=r"^DSC_"),
    ],
    mode="or",
)
matches = [e for e in all_entries if combined_or(e)]
print(f"OR filter (IMG_ or DSC_ prefix): {len(matches)} matches")

## 5. Group Files by Date

In [ ]:
from tidyforge.media_grouping import ByDate, ByExtension, GroupingEngine

# Group by month
engine = GroupingEngine(ByDate(granularity="month"))
summaries = engine.group_summary(all_entries)

print(f"{len(summaries)} date groups:\n")
for s in summaries:
    exts = ", ".join(sorted(s.extensions))
    print(f"  {s.key:10s}  {s.count:4d} files  {s.size_human:>8s}  exts: {exts}")

In [ ]:
# Group by extension
engine_ext = GroupingEngine(ByExtension())
summaries_ext = engine_ext.group_summary(all_entries)

print(f"{len(summaries_ext)} extension groups:\n")
for s in summaries_ext:
    print(f"  {s.key:8s}  {s.count:4d} files  {s.size_human:>8s}")

## 6. Categorise Files

In [ ]:
from collections import Counter

from tidyforge.metadata import categorize_file

categories = Counter(categorize_file(e.path) for e in all_entries)
print("Category breakdown:\n")
for cat, count in categories.most_common():
    print(f"  {cat:12s}  {count} files")

## 7. Find Duplicates

In [ ]:
from tidyforge.duplicate_detection import find_duplicates

paths = [e.path for e in all_entries]
report = find_duplicates(paths)

print(f"{len(report.groups)} duplicate groups found\n")
for group in report.groups[:5]:
    print(f"  Hash: {group.hash[:12]}...  Size: {group.size} bytes")
    for p in group.paths:
        print(f"    {p}")
    print()